# Contract Review Assistant — package client

This notebook calls the extracted package.
It does not redefine the model or preprocessing.

Run from a clean kernel, in order. Outputs contain aggregate results only.


In [ ]:
import os
from pathlib import Path

import pandas as pd

from contract_review.evaluation import eval_scores
from contract_review.pipeline import run_configured_pipeline
from contract_review.review import pack_eval

current = Path.cwd().resolve()
root = next(
    path for path in (current, *current.parents)
    if (path / "pyproject.toml").exists()
)
os.chdir(root)


In [ ]:
result, settings = run_configured_pipeline()


In [ ]:
evaluation = eval_scores(
    result["combined_scores"],
    result["masks"]["test"],
    result["chunks"],
    result["labels"],
    result["categories"],
)

evaluation[["AUPR", "P@3", "R@3"]].mean().to_frame("test")


In [ ]:
import json

red_flags = json.loads(
    Path(settings["red_flags_path"]).read_text(encoding="utf-8")
)

packs = pd.concat([
    pack_eval(
        result["probabilities"],
        result["contracts"],
        result["chunks"],
        result["labels"],
        result["categories"],
        red_flags,
        split,
        budgets=(settings["review_budget"],),
    )
    for split in ["valid", "test", "held-out type"]
])

summary = packs.groupby("split")[
    ["present", "found", "highlights", "on target"]
].sum()

summary["red-flag recall"] = summary["found"] / summary["present"]
summary["highlights on target"] = (
    summary["on target"] / summary["highlights"]
)
summary
